# Drone Detection & Tracking — end to end on Kaggle T4×2

Trains **YOLO11n** — the edge candidate — on **Drone-vs-Bird (`dvb`)** across both GPUs,
and evaluates it.

This is the whole ML pipeline in one run: acquire → convert → split → build → verify →
train → score. Nothing in this repository downloads or trains by itself — every stage
below is an explicit command, and `sanity` is a hard gate that stops the run before it can
waste GPU hours on a leaked split.

### Why `dvb`, and why alone

`dvb` is the **only** dataset in the matrix that can *falsify* a false-positive claim.
Three of the four sources are drone-only tracking benchmarks: they contain no bird
annotations at all, so a detector that labelled every bird as a drone would score
perfectly on them. `dvb` has genuine bird negatives beside the drone class, which makes its
precision number *mean* something. It is also the dataset the deployed system exists to get
right — the failure mode it prevents is a bird being called a drone, and a page being raised
for nothing.

### What you should see

| Stage | Command | Approx. cost |
| --- | --- | --- |
| Verify | `anti-uav verify-env` | seconds |
| Weights | `anti-uav fetch-weights` | ~6 MB |
| Data | `anti-uav download --dataset dvb` | ~6.7 GB |
| Preprocess | `convert` → `splits` → `build` → `sanity` | ~15–25 min |
| Train | `anti-uav train --model yolo11n` | **1.5–3 h** (120 epochs) |
| Evaluate | `anti-uav evaluate --cross` | ~2 min |

The training cell dominates. If you only have an hour, cut `--epochs` to 30 — the mAP is
still meaningful well before 120 and you get the whole pipeline on the record.

---

## Before you run

**Settings → Accelerator → GPU T4 x2.** Kaggle restarts the session when you change this,
so do it before running anything.

**Settings → Internet → On.** Required three times over: the `git clone`, the `uv` install,
and the Kaggle dataset download.

**Add a Kaggle API token as a notebook secret** (Settings → Secrets, or Add-ons →
Secrets). One of these works:

| Secret name | Value |
| --- | --- |
| `KAGGLE_API_TOKEN` | the token from Account → Settings → API — **preferred** |
| `KAGGLE_USERNAME` + `KAGGLE_KEY` | account name and legacy API key, both required |

`KAGGLE_API_TOKEN` is the modern path and the one Kaggle's UI hands you by default. The
username/key pair is the older shape and still authenticates, but both halves must be set —
one alone is not a credential, and the download will refuse with a message naming what is
missing rather than handing you an opaque 401 from the CLI.

The notebook has no credentials of its own. They are read from secrets into the environment
in the download step and never written to a file that gets committed.

One Kaggle quirk worth knowing in advance: you may also need to **accept the rules** for
`romsham/dronevsbird-foryolo` in a browser tab. Kaggle gates that dataset behind a rules
acceptance that the API cannot satisfy for you, and the error it returns is a 403 with no
explanation.

---

In [ ]:
# --- 1. Clone -------------------------------------------------------------

REPO_URL = "https://github.com/uzairlol/drone-detection-tracking.git"
REPO_DIR = "/kaggle/working/anti-uav"

!git clone --depth 1 {REPO_URL} {REPO_DIR}

In [ ]:
# --- 2. Install -----------------------------------------------------------

# `uv` rather than `pip`: same resolution, a fraction of the wall clock, and it leaves an
# already-satisfied requirement alone. That last part is the point - Kaggle ships a
# CUDA-matched torch/numpy pair, and replacing it with a different build is the classic way
# to break `import torch` on a GPU that works fine.
#
# `--system` installs into the image's interpreter. A venv would need its bin/ prepended to
# PATH in every subsequent cell, which is one forgotten cell away from a silent ImportError.
#
# `pip install -e ".[train,download]"` is an exact substitute if you would rather not add
# a dependency manager; it is only slower.

!pip install --quiet uv
!uv pip install --system -e "{REPO_DIR}[train,download]"

### Reading the slow cells

Every command after this one runs through a small `run()` helper instead of a bare `!` shell
escape. It does three things worth having:

- **Echoes the command first**, so the cell you are looking at and the command that actually
  ran cannot be different.
- **Streams output line by line.** A 2-hour training run is then visible progress rather than
  a spinning indicator, and ultralytics' per-epoch mAP is readable as it climbs.
- **Reports wall clock, including on failure** — `[ok] 1h 42m` or `[EXIT 1] 6m 03s`. Knowing
  how far it got before dying is usually the most useful thing in the output.

`build` and `sanity` are also *gated* now: the cell raises and stops rather than carrying on
into a training run with a leaked split.

### Scratch

Everything writable goes to `/kaggle/tmp`, never `/kaggle/working`.

This is a hard limit, not a preference: only **20 GB** of `/kaggle/working` is autosaved per
notebook version, while the raw dataset plus the interim and processed trees come to ~15 GB
and then some. `/kaggle/tmp` is scratch — far larger, and discarded with the session.

`ANTI_UAV_DATA_DIR` moves the `data/` and `artifacts/` trees while `configs/` is still read
from the clone, so not one pipeline behaviour changes. It must be set **before** the first
`anti-uav` call, because path resolution is cached on first use.

Then read `verify-env` rather than trusting the exit code alone. Three lines carry the
meaning:

| Line | Must say | If it does not |
| --- | --- | --- |
| `profile` | `turing` | `pascal` means auto-detection did not run and **AMP is off** |
| `python` | `[ok]` | Kaggle ships 3.13, which is supported. 3.14+ is refused as untested |
| `arch-kernels` | `[ok] … sm_75 present` | Kaggle's torch lacks Turing kernels; nothing else will help |
| `base-weights` | warn, not fail | expected — `*.pt` is gitignored; cell 5 fetches it |

Two GPUs is also worth confirming here: you should see two `device` / `arch-kernels` pairs,
one per T4. If only one appears you have a single-GPU session, and `--device 0,1` will fail.

This step exists because the failure it catches is silent. A torch wheel built without your
card's architecture imports cleanly, reports `torch.cuda.is_available() == True`, and then
dies hours in with `no kernel image is available for execution on the device`. Checking
availability is not enough; the arch list has to be checked too.

In [ ]:
import os

os.environ["ANTI_UAV_DATA_DIR"] = "/kaggle/tmp/anti-uav"

# Auto-detect is correct on a T4 now (sm_75 -> turing). Pinned anyway, so the notebook cannot
# silently pick a different profile if the accelerator is switched to a P100 (sm_60).
os.environ["ANTI_UAV_PROFILE"] = "turing"

# A small runner for the slow cells. It prints the command first, streams its output so a
# long run is not a silent spinning dot, and reports wall clock either way - including when
# something fails, which is when you most want to know how far it got before dying.
import shutil
import subprocess
import time

def run(*argv, note=""):
    """Run a CLI command, streaming output, with a wall-clock report."""
    import shutil as _shutil

    if _shutil.which(argv[0]) is None:
        raise SystemExit(
            f"{argv[0]!r} is not on PATH. If this is the anti-uav command, the\n"
            f"editable install did not land - re-run the install cell."
        )
    print(f"\n\033[1m$ {' '.join(argv)}\033[0m")
    if note:
        print(f"  {note}")
    started = time.monotonic()
    proc = subprocess.Popen(argv, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, env=os.environ.copy())
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    elapsed = time.monotonic() - started
    mins, secs = divmod(elapsed, 60)
    verdict = "ok" if proc.returncode == 0 else f"EXIT {proc.returncode}"
    print(f"\033[1m  [{verdict}] {mins:.0f}m {secs:04.1f}s\033[0m")
    return proc.returncode

def anti_uav(*args, note=""):
    return run("anti-uav", *args, note=note)

# Proves the environment cell actually wrote through to a subprocess, rather than
# only setting a Python-local. Catches forgetting this cell before a 6.7 GB download.
print("ANTI_UAV_DATA_DIR ->", os.environ["ANTI_UAV_DATA_DIR"])
print("anti-uav on PATH ->", shutil.which("anti-uav") or "NOT FOUND")

anti_uav("verify-env", note="expect: profile turing, 2x sm_75, and no FAIL")

## 4. Acquire

**Weights first, on purpose.** ultralytics downloads a missing base checkpoint from *inside*
the trainer, so on a slow or blocked network the run dies *after* the dataset is built and the
run directory created. Fetching here moves that failure to a five-second one.

Then `dvb`, from the public Kaggle mirror. `--dry-run` first every time: it prints exactly
what would be fetched, from where, and how large — and it is the cheapest way to catch a
missing credential before committing to a 6.7 GB transfer.

In [ ]:
# Prefer KAGGLE_API_TOKEN; fall back to the username/key pair. Whichever is
# present, `download` names the shape it found rather than just saying yes/no.
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
_loaded = []

for _name in ("KAGGLE_API_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"):
    try:
        os.environ[_name] = _secrets.get_secret(_name)
        _loaded.append(_name)
    except Exception:
        pass

print(f"credentials loaded: {', '.join(_loaded) or 'NONE'}")

anti_uav("fetch-weights", "--models", "yolo11n", note="~6 MB")

In [ ]:
anti_uav("download", "--dataset", "dvb", "--dry-run",
         note="free - prints what would be fetched and whether credentials resolved")

In [ ]:
# ~6.7 GB. Streams the CLI's progress as it goes.
#
# EXPECT TWO RESULT LINES, and they mean different things:
#   completed           the Kaggle mirror - this is your training data
#   reference only      the WOSDETC annotations - never fetched, and that is fine.
#                       It is a cross-check aid, not a training input, so nothing is
#                       missing. Do not go looking for a login to satisfy it.
anti_uav("download", "--dataset", "dvb", note="~6.7 GB, several minutes")

## 5. Preprocess — and why the order is not negotiable

```
convert  →  splits  →  build  →  sanity
```

`build` reads each frame's split field off the interim index. Build before splitting and you
materialise an **empty train set**, then `sanity` fails — correctly, but after wasting the
copy.

**`splits` assigns whole sequences, never individual frames.** All four datasets here are
video-derived, so adjacent frames are near-identical. Split at the frame level and the same
target's near-duplicates land on both sides of the boundary: mAP inflates by double digits and
the number is meaningless. `sanity` fails the build if any sequence appears in two splits, so
this is enforced rather than merely documented.

**`convert` reads in place.** The ~6.7 GB of PNGs is never copied into the working tree.

### What `stats` will tell you

Expect roughly **17.9k frames**, ~1.15 boxes each, and a bird-to-drone box ratio near 1.3.

The converter checks that ratio against the published statistics and **aborts** above 1.5.
This is not paranoia: the Kaggle mirror renames classes to bare integers, and a silent 0/1
inversion is invisible in a per-class mAP table unless you already know which way round it
should be — while inverting the one output the whole system depends on.

One surprise to expect: **`dvb` is configured as tiled at 640 px**, because its median target
(28 px) is below the 40 px threshold. But this mirror already ships 640×640 crops, so
`tile_grid` returns exactly one tile per image and tiling is a no-op. That is why this run is
~17.9k frames rather than the 4–6× the docs quote for a 4K source. The tiling path is still
exercised — it simply has nothing to do.

In [ ]:
anti_uav("convert", "--dataset", "dvb", note="~18k frames, several minutes")
anti_uav("stats", "--dataset", "dvb", note="~1 min")

In [ ]:
# convert -> splits -> build -> sanity. The order is load-bearing: `build` reads each
# frame's split field off the interim index, so building first gives an empty train set.
# `sanity` exits non-zero on failure - read its output, do not just scroll past it.

# `splits` is seconds. `build` is the slow one: it re-encodes every frame into the YOLO
# layout, so it writes ~18k images and is where the 15-25 min goes.
if anti_uav("splits", "--combo", "dvb", note="seconds") != 0:
    raise SystemExit("splits failed - do not continue, build would produce an empty train set")

anti_uav("build", "--combo", "dvb", note="~15-25 min, writes ~18k images")

# Last gate. Non-zero here means a leaked split or a failed invariant - stop.
if anti_uav("sanity", "--combo", "dvb", note="hard gate") != 0:
    raise SystemExit("sanity FAILED - do not train on this build")

## 6. Train

Dry-run first, every time. It prints the resolved hyperparameters, the output path and the
dataset summary without allocating a GPU — the cheapest possible way to catch a wrong combo,
and the only place the resolved batch is visible before it either works or OOMs.

```
batch  : 32        ← a concrete integer; a fraction here aborts under DDP
amp    : True      ← if False, the turing profile did not load
```

### Reading the batch under DDP

The printed batch is the **total across both GPUs**, so 32 means 16 per rank. It is a fixed
integer rather than AutoBatch for two reasons: ultralytics resolves `batch: -1` to a
*fraction* of free VRAM and then raises outright when `world_size > 1`, and DDP splits the
total evenly, so an odd number would leave a rank idle.

### Performance expectation

**~1.5–3 h for 120 epochs on T4×2.** Expect a **sub-2× speedup from two GPUs**, not 2×: at
this dataset size the bottleneck is four CPU cores decoding 640 px JPEG, not the cards. These
are order-of-magnitude estimates from FLOP counts, not measurements.

This is a good box for proving the pipeline end to end; the full 14-run matrix belongs on the
lab's RTX 4090/5090. **If you OOM, halve the batch before changing anything else.**

In [ ]:
anti_uav("train", "--model", "yolo11n", "--combo", "dvb",
         "--profile", "turing", "--device", "0,1", "--workers", "2",
         note="dry run: confirms batch 32 / amp True before committing GPU hours")

### YOLO11n — the edge candidate

2.6 M parameters, which is what makes it the deployable option: small enough to sit behind
`nvinfer` on a Jetson against a ~3 ms budget. The matrix's other family, RT-DETR-x2, is the
accuracy reference and is not run here.

In [ ]:
# ~1.5-3 h for 120 epochs. Add --epochs 30 for a fast pass.
#
# FALLBACK: if DDP cannot spawn inside the notebook kernel, re-run with --device 0
# (single GPU, same batch). Identical results, similar wall clock, no subprocess to
# go wrong. The preflight warns before training if you ask for more GPUs than are present.
#
# This is the long one. `run()` streams ultralytics' per-epoch lines, so you can watch
# it progress and see mAP climb rather than staring at a spinning indicator for two
# hours. Each epoch should take ~45-90 s; if the first one takes minutes, something is
# wrong (falling back to one GPU) and it is far better to notice now than in 90 minutes.
anti_uav(
    "train", "--model", "yolo11n", "--combo", "dvb",
    "--profile", "turing", "--device", "0,1", "--workers", "2",
    note="1.5-3 h for 120 epochs - this is the one to leave running",
)

### Reading the run

`list-runs` reads the run directory off disk and prints run, model, epochs, mAP50,
mAP50-95, precision, recall and wall clock.

**mAP50-95 is the headline, not mAP50.** At a 28 px median target, mAP50 is generous — a
detector that puts a loose box on the right patch scores well. The stricter metric is the one
that discriminates between a detector that found the bird and one that found a patch of sky.

## 7. Evaluate

`--cross` is the number worth showing. A single combined mAP across heterogeneous datasets is
close to meaningless, because the sources disagree about what a hard sample is — MM-UAV's
12 px targets dominate the loss while Drone-vs-Bird's birds dominate the precision. The
per-dataset table is the honest one.

### Read the falsifiability note, not just the number

The evaluator prints a caveat next to any precision figure whose dataset cannot support the
claim:

> for `antiuav` and `mmuav`, precision cannot falsify a false-positive claim — neither
> contains bird annotations, so a detector that called every bird a drone would score
> perfectly.

On `dvb` it **can**. That is the entire reason this dataset is the demo, and it is the one
number in this notebook that can actually be *wrong* in an informative way.

One caveat specific to this run: `dvb` has **no negative-only frames** — every frame contains
at least one labelled object. A detector trained mostly on positives never learns a usable
confidence floor, so `rules.confidence.initiate` has nothing calibrated against it. The
`stats` output flags this. It is a property of the source data, not of the pipeline.

In [ ]:
anti_uav("evaluate", "--run", "artifacts/runs/yolo11n/dvb", "--cross",
         note="~2 min")

## 8. Why `track-eval` refuses this dataset

Run it and it declines. **That is correct behaviour, and worth demonstrating rather than
working around** — it is the tooling refusing to report a number it cannot stand behind.

| Dataset | Bird negatives | MOT ground truth | Multi-object identity | Verdict |
| --- | --- | --- | --- | --- |
| `dvb` | **yes** | no | no | detection metrics only |
| `mavvid` | **yes** | no | no | detection metrics only |
| `antiuav` | no | yes | **no — single target** | MOTA/HOTA only; IDF1 and IDSW are vacuous |
| `mmuav` | no | yes | **yes** | everything, including IDF1 |

The subtle row is `antiuav`. It *has* track ids, so a naive `has_track_ids` check passes —
but every box is id 1, so every identity trivially matches every other. IDF1 would report a
perfect score for a tracker doing nothing. That is why `capabilities.py` derives an explicit
contract per source rather than trusting a single boolean, and why `track-eval` **warns** on
single-target ground truth instead of refusing it.

Only `mmuav` makes IDF1 meaningful, and it is Baidu-Pan-only with an interactive captcha —
not something to attempt live. The trackers themselves run fine on `dvb`; there is simply
nothing to score them against.

In [ ]:
anti_uav("track-eval", "--run", "artifacts/runs/yolo11n/dvb", "--dataset", "dvb",
         note="declines by design: dvb has no track ids")

## 9. Inference and the rule layer

**This project's own trackers, not ultralytics'.** Version 8.4 ships no `track` task —
`persist` and `tracker` are not in the predict config, and `model.predict(..., persist=True)`
raises `SyntaxError`. Owning SORT / ByteTrack / BoT-SORT buys something more valuable: the
live path and the `track-eval` harness share **one** association implementation, so a number
from the harness describes what the pipeline actually does.

### Why inference is permissive and alerting is strict

Two different stages, two different jobs:

| | threshold | why |
| --- | --- | --- |
| detector | `conf_threshold: 0.25` | maximise **recall** — the tracker needs the low-confidence frames to carry motion evidence through a dip |
| rules | `confidence.initiate: 0.60` | maximise **precision** — buy it *after* tracking, using persistence, kinematics and cross-camera agreement |

Raise the detector threshold to 0.60 and the tracker starves. `predict` therefore prints the
per-detection table rather than a count, because 40 detections at conf 0.25 and 3 at 0.8 are
completely different situations.

### The five rule gates

`configs/rules/drone_rules.yaml`, in order: **confidence** (with hysteresis), **persistence**,
**kinematics**, **spatial**, **cross-camera**. Every threshold is tunable without a retrain.

Two of them return `skipped` rather than `passing` until the site is calibrated —
`spatial.horizon_y` and `ground_plane_z_m` are null, and they are per-camera properties.
That distinction is the point of the layer: **a missing input must never read as a satisfied
rule.** Only `FAIL` blocks an alert; an uncalibrated site runs three of five gates and says
plainly which two it skipped and what to configure.

Note `cross_camera.min_cameras: 2`. One view cannot distinguish a drone from a bird with
enough confidence to page anyone — so a track with no global identity is a **failure**, not a
skip. This is why cross-camera fusion is load-bearing for correctness rather than a nice
extra.

In [ ]:
# Grab a real frame from the interim index so the cell is self-contained. Swap --source for
# a video clip to show tracker behaviour over time - that is the more convincing demo, since
# a single frame cannot show a track being held through a dip or broken by an occlusion.
import glob

frames = sorted(
    glob.glob(f"{os.environ['ANTI_UAV_DATA_DIR']}/data/interim/dvb/*/frames/*/rgb/*.jpg")
)
print(f"{len(frames):,} interim frames available")

anti_uav("predict", "--run", "artifacts/runs/yolo11n/dvb",
         "--source", frames[len(frames) // 2], "--tracker", "botsort",
         "--save", "/kaggle/tmp/preds", note="seconds")

In [ ]:
anti_uav("rules", "show")
anti_uav("rules", "validate", note="cross-rule tensions; advisory")

## 10. Keep the evidence

`artifacts/` lives under `/kaggle/tmp` and is **not** autosaved — the weights would vanish with
the session. Copy the record out to `/kaggle/working`, which is. That single step is the
difference between a demo and a result you can point at afterwards.

A run directory is the record: `best.pt`, `results.csv`, `args.yaml` and a metadata file.
**Copy it, do not regenerate it.** `data/processed/dvb/build_report.json` is worth keeping too
— it is what proves which sources were actually present, and `build` will happily skip a
missing one and still name the combo after it. A combo name is not evidence of dataset
membership.

In [ ]:
import shutil
from pathlib import Path

SRC = Path(os.environ["ANTI_UAV_DATA_DIR"])
DST = Path("/kaggle/working/artifacts")

for _rel in ("runs", "exports", "tracks"):
    _src = SRC / "artifacts" / _rel
    if _src.exists():
        shutil.copytree(_src, DST / _rel, dirs_exist_ok=True)
        print(f"saved artifacts/{_rel}")

_report = SRC / "data" / "processed" / "dvb" / "build_report.json"
if _report.is_file():
    (DST / "dvb_build_report.json").write_text(_report.read_text())
    print("saved dvb_build_report.json")

In [ ]:
anti_uav("list-runs")

## What this notebook deliberately does not do

**`anti-uav serve`** — the operator console and JSON API. Kaggle tears the session down when
the last cell finishes and does not forward a listening port, so a long-lived server has
nowhere to live. Run it locally against the copied `artifacts/` — the UI reads runs off disk,
so the copied record is all it needs.

**TensorRT export** — the engine is normally built on the target device, not on a training
box, so `deploy render` *warns* rather than fails when the engine is missing and prints the
exact `trtexec` command. ONNX export works if you want it.

**RT-DETR-x2** — the matrix's accuracy reference, at roughly 4–6 h on T4×2. Not run here;
the recipe, the per-model batch and the evaluator path are all in place for it
(`anti-uav train --model rtdetr_x2 --combo dvb --profile turing --device 0,1`).

**The other 13 runs** — the matrix is 2 models × 7 combos; this notebook does 1 of the 14.
T4×2 proves the pipeline; it does not sweep it. On the 5090 the full matrix is hours; here it
is days.

**Tracking metrics** — see section 8. Only `mmuav` makes them meaningful, and it is
Baidu-Pan-only behind an interactive captcha.

---

## What one run does and does not establish

Four public datasets, one architecture, one combo, one site model. This run answers **"does
the pipeline work end to end, and how well does the detector reject birds on the one dataset
that can test that"**.

It does not answer "how well does this detect an unseen drone type at 40 m against a cluttered
sky". Four datasets is a design choice made under a time budget, not a claim of coverage.
**Treat the ranking as a ranking and the absolute numbers as specific to these four sources.**

The strongest claim here is a negative one, which is the useful kind: whether the detector
fires on birds. Everything finer needs the full matrix on real hardware.